# Verificação da nova codificação de `ALRM_*` e `GRAV_*`

O que muda nos 24 campos clínicos quando o ausente deixa de chegar ao modelo como 0.
Gera e confere as bases das três configurações de preditores definidas em `utils/configuracoes.py`: original, final e sem critérios de gravidade.
Lê `data/raw/dengue_hospitalized.parquet` e, para a verificação de identidade, `data/processed/dengue_treated.parquet`, e grava uma base por configuração nova em `data/processed/`.
Nenhum modelo é treinado e nenhuma métrica de desempenho é calculada.

## Etapas
1. Aplicar `treat()` nas duas codificações sobre a mesma base bruta.
2. Comparar a base da codificação `antiga`, coluna a coluna, com a base tratada em produção.
3. Medir os ausentes campo a campo nos 24 campos clínicos e a distribuição das colunas de bloco avaliado.
4. Conferir as colunas de bloco contra os indicadores reconstruídos como no notebook 06.
5. Descrever a composição das três configurações e a pertinência campo a campo.
6. Analisar o código 10 da escolaridade por partição e sua relação com a idade.
7. Gravar as bases das duas configurações novas e conferir a matriz de desenho por sha256.
8. Medir a ausência e o EPP por configuração na janela de treino.

## Entradas
- `data/raw/dengue_hospitalized.parquet`
- `data/processed/dengue_treated.parquet`
- `data/features/baseline/config.json`

## Saídas
- `data/processed/dengue_treated_corrigida.parquet`
- `data/processed/dengue_treated_notificacao.parquet`
- `analysis/results_codificacao/identidade_base_antiga.csv`
- `analysis/results_codificacao/ausentes_por_campo_base_corrigida.csv`
- `analysis/results_codificacao/distribuicao_colunas_bloco_avaliado.csv`
- `analysis/results_codificacao/concordancia_com_notebook_06.csv`
- `analysis/results_codificacao/configuracoes_preditores.csv`
- `analysis/results_codificacao/pertinencia_campos_por_configuracao.csv`
- `analysis/results_codificacao/escolaridade_codigo_10_por_particao.csv`
- `analysis/results_codificacao/idade_por_grupo_escolaridade.csv`
- `analysis/results_codificacao/codigo_10_e_idade.csv`
- `analysis/results_codificacao/bases_geradas.csv`
- `analysis/results_codificacao/matriz_desenho_antes_e_depois_da_correcao.csv`
- `analysis/results_codificacao/comparativo_configuracoes.csv`
- `analysis/results_codificacao/epp_por_configuracao.csv`

Importações, caminhos, formatadores em português e as constantes dos dois blocos.

In [1]:
import os
import sys
import json
import hashlib
import textwrap

import numpy as np
import pandas as pd

# Raiz do repositorio por marcador estavel, e nao por contagem de niveis:
# reorganizar as pastas de notebooks deixa de quebrar os caminhos.
def _raiz_repo(inicio=None):
    d = os.path.abspath(inicio or os.getcwd())
    while not any(os.path.exists(os.path.join(d, m)) for m in ('pyproject.toml', '.git')):
        pai = os.path.dirname(d)
        if pai == d:
            raise RuntimeError(f'raiz do repositorio nao encontrada acima de {os.getcwd()}')
        d = pai
    return d

ROOT = _raiz_repo()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from utils import clean_data, treat_data, split_data, configuracoes, preprocessamento

RAW_PATH      = os.path.join(ROOT, 'data', 'raw', 'dengue_hospitalized.parquet')
TREATED_ATUAL = os.path.join(ROOT, 'data', 'processed', 'dengue_treated.parquet')
PROC_DIR      = os.path.join(ROOT, 'data', 'processed')
CFG_PATH      = os.path.join(ROOT, 'data', 'features', 'baseline', 'config.json')
OUT_TAB       = os.path.join(ROOT, 'analysis', 'results_codificacao')
os.makedirs(OUT_TAB, exist_ok=True)

RANDOM_STATE = 42


def salvar_tab(df, nome, index=False):
    caminho = os.path.join(OUT_TAB, nome)
    df.to_csv(caminho, index=index, encoding='utf-8')
    print(f'[tabela] {os.path.relpath(caminho, ROOT)}  ({len(df)} linhas)')
    return df


def n_(x):
    '''Inteiro com ponto de milhar.'''
    return f'{int(x):,}'.replace(',', '.')


def v_(x, casas=1):
    '''Decimal com virgula.'''
    return f'{x:.{casas}f}'.replace('.', ',')


def par_(texto):
    '''Paragrafo do resumo, quebrado em 95 colunas como o resto do arquivo.'''
    return textwrap.fill(' '.join(texto.split()), width=95)


# Chave interna -> rotulo publicado. As chaves ('antiga', 'corrigida', 'notificacao')
# sao identificadores e nomeiam os parquets; os rotulos sao o que aparece em tabela,
# figura e prosa.
ROTULO_CONFIG = {
    'antiga':      'original',
    'corrigida':   'final',
    'notificacao': 'sem critérios de gravidade',
}
LARG_ROTULO = max(len(r) for r in ROTULO_CONFIG.values())

ALRM_COLS = treat_data.ALRM_COLS
GRAV_COLS = treat_data.GRAV_COLS
BLOCO_ALARME, BLOCO_GRAVIDADE = treat_data.BLOCO_COLS

with open(CFG_PATH) as f:
    cfg_publicado = json.load(f)

print(f'Codificações aceitas por treat(): {treat_data.CODIFICACOES}')
print(f'Configurações disponíveis: {configuracoes.NOMES}')
print(f'Blocos: {len(ALRM_COLS)} campos de alarme, {len(GRAV_COLS)} de gravidade')

Codificações aceitas por treat(): ('antiga', 'corrigida')
Configurações disponíveis: ('antiga', 'corrigida', 'notificacao')
Blocos: 9 campos de alarme, 15 de gravidade


Aplica `treat()` nas duas codificações sobre a mesma base bruta.

In [2]:
bruto = pd.read_parquet(RAW_PATH, engine='pyarrow')
print(f'{os.path.relpath(RAW_PATH, ROOT)}: {len(bruto):,} registros')

base_antiga    = treat_data.treat(bruto.copy(), codificacao='antiga')
base_corrigida = treat_data.treat(bruto.copy(), codificacao='corrigida')

print(f"treat(codificacao='antiga'):    {base_antiga.shape[0]:,} x {base_antiga.shape[1]}")
print(f"treat(codificacao='corrigida'): {base_corrigida.shape[0]:,} x {base_corrigida.shape[1]}")
print(f'Colunas novas em corrigida: '
      f'{[c for c in base_corrigida.columns if c not in base_antiga.columns]}')

data/raw/dengue_hospitalized.parquet: 350,850 registros


treat(codificacao='antiga'):    350,850 x 63
treat(codificacao='corrigida'): 350,850 x 66
Colunas novas em corrigida: ['bloco_alarme_avaliado', 'bloco_gravidade_avaliado', 'macrorregiao']


Compara a base da codificação antiga, coluna a coluna, com a base tratada em produção. A correção do passo 6 de `treat_data.py` faz as duas divergirem num ponto conhecido: o ausente de uma coluna `object` deixou de virar o texto `'nan'`. A verificação separa esse efeito de qualquer outro e interrompe se encontrar divergência em valor presente.

In [3]:
atual = pd.read_parquet(TREATED_ATUAL, engine='pyarrow')

divergencias = []

if list(atual.columns) != list(base_antiga.columns):
    divergencias.append({'tipo': 'conjunto/ordem de colunas', 'coluna': '—',
                         'detalhe': f'atual {len(atual.columns)} vs antiga {len(base_antiga.columns)}'})
if len(atual) != len(base_antiga):
    divergencias.append({'tipo': 'numero de linhas', 'coluna': '—',
                         'detalhe': f'atual {len(atual):,} vs antiga {len(base_antiga):,}'})


def so_ausencia_estringada(a, b, dif):
    '''True se toda celula divergente e ausente dos dois lados: texto num, NaN no outro.

    E a assinatura da correcao do passo 6. Qualquer outra divergencia -- valor presente
    que mudou, ausente que virou presente -- devolve False e derruba a verificacao.
    '''
    texto_de_ausente = {'nan', 'None', 'NaT', '<NA>'}
    antes = a[dif]
    depois = b[dif]
    return bool(antes.astype(str).isin(texto_de_ausente).all() and depois.isna().all())


comuns = [c for c in atual.columns if c in base_antiga.columns]
linhas_cmp = []
for c in comuns:
    a, b = atual[c], base_antiga[c]
    mesmo_tipo = str(a.dtype) == str(b.dtype)
    # equals() trata NaN == NaN como igual, que e o comportamento desejado aqui
    iguais = a.reset_index(drop=True).equals(b.reset_index(drop=True))
    dif = (~((a.values == b.values) | (a.isna().values & b.isna().values))
           if not iguais else np.zeros(len(a), dtype=bool))
    n_dif = int(dif.sum())
    passo_6 = bool(n_dif and so_ausencia_estringada(a, b, dif))
    linhas_cmp.append({'coluna': c, 'dtype_atual': str(a.dtype), 'dtype_novo': str(b.dtype),
                       'mesmo_dtype': mesmo_tipo, 'identica': iguais, 'n_celulas_diferentes': n_dif,
                       'divergencia': ('nenhuma' if iguais else
                                       'correção do passo 6' if passo_6 else 'outra'),
                       'n_ausentes_atual': int(a.isna().sum()),
                       'n_ausentes_novo': int(b.isna().sum()),
                       'n_base': len(atual)})
    if (not iguais and not passo_6) or not mesmo_tipo:
        divergencias.append({'tipo': 'conteudo/dtype', 'coluna': c,
                             'detalhe': f'identica={iguais} mesmo_dtype={mesmo_tipo} n_dif={n_dif}'})

comparacao = pd.DataFrame(linhas_cmp)
salvar_tab(comparacao, 'identidade_base_antiga.csv')

passo_6 = comparacao[comparacao['divergencia'] == 'correção do passo 6']
print(f'Colunas comparadas: {len(comuns)} | idênticas: {int(comparacao["identica"].sum())} '
      f'| dtypes iguais: {int(comparacao["mesmo_dtype"].sum())}')
print(f'Linhas: atual {len(atual):,} | antiga {len(base_antiga):,}')

if divergencias:
    display(pd.DataFrame(divergencias))
    raise AssertionError(
        f'PARADA: a base antiga divergiu da atual em {len(divergencias)} ponto(s) fora da '
        'correção do passo 6. Ver a tabela acima e identidade_base_antiga.csv.'
    )

if len(passo_6):
    display(passo_6[['coluna', 'n_celulas_diferentes', 'n_ausentes_atual', 'n_ausentes_novo']])
    for _, r in passo_6.iterrows():
        print(f'{r["coluna"]}: {n_(r["n_celulas_diferentes"])} células de ausente que a base '
              f'em produção guarda como texto e a nova guarda como NaN; ausentes detectáveis '
              f'por isna() passam de {n_(r["n_ausentes_atual"])} para {n_(r["n_ausentes_novo"])}.')
    print('\nNenhum valor presente mudou: a única divergência é o ausente que o astype(str) '
          'do passo 6 convertia em texto.')
else:
    print('\nNenhuma divergência: a codificação padrão preserva integralmente o '
          'comportamento atual.')

[tabela] analysis/results_codificacao/identidade_base_antiga.csv  (63 linhas)
Colunas comparadas: 63 | idênticas: 62 | dtypes iguais: 63
Linhas: atual 350,850 | antiga 350,850


,coluna,n_celulas_diferentes,n_ausentes_atual,n_ausentes_novo
0,CS_SEXO,187,2,189


CS_SEXO: 187 células de ausente que a base em produção guarda como texto e a nova guarda como NaN; ausentes detectáveis por isna() passam de 2 para 189.

Nenhum valor presente mudou: a única divergência é o ausente que o astype(str) do passo 6 convertia em texto.


Ausentes campo a campo nos 24 campos clínicos da base da configuração final.

In [4]:
N = len(base_corrigida)
alvo = base_corrigida['target']
n_obito = int((alvo == 1).sum())

print(f'Base corrigida: {N:,} linhas x {base_corrigida.shape[1]} colunas '
      f'| óbitos {n_obito:,} ({n_obito / N * 100:.2f}%)')

campos_24 = ALRM_COLS + GRAV_COLS
nan_24 = []
for c in campos_24:
    n_nan = int(base_corrigida[c].isna().sum())
    nan_24.append({
        'campo': c,
        'bloco': 'alarme' if c in ALRM_COLS else 'gravidade',
        'n_ausente': n_nan,
        'pct_ausente': round(n_nan / N * 100, 2),
        'n_valor_1_sim': int((base_corrigida[c] == 1).sum()),
        'n_valor_0_nao': int((base_corrigida[c] == 0).sum()),
        'n_base': N,
    })
nan_24 = pd.DataFrame(nan_24)
salvar_tab(nan_24, 'ausentes_por_campo_base_corrigida.csv')
display(nan_24)

Base corrigida: 350,850 linhas x 66 colunas | óbitos 9,451 (2.69%)
[tabela] analysis/results_codificacao/ausentes_por_campo_base_corrigida.csv  (24 linhas)


,campo,bloco,n_ausente,pct_ausente,n_valor_1_sim,n_valor_0_nao,n_base
0,ALRM_HIPOT,alarme,242914,69.24,20891,87045,350850
1,ALRM_PLAQ,alarme,242632,69.16,73933,34285,350850
2,ALRM_VOM,alarme,242957,69.25,21495,86398,350850
3,ALRM_SANG,alarme,242908,69.23,27122,80820,350850
4,ALRM_HEMAT,alarme,243117,69.29,8141,99592,350850
5,ALRM_ABDOM,alarme,242823,69.21,43187,64840,350850
6,ALRM_LETAR,alarme,243047,69.27,11803,96000,350850
7,ALRM_HEPAT,alarme,243210,69.32,2318,105322,350850
8,ALRM_LIQ,alarme,243149,69.30,5251,102450,350850
9,GRAV_PULSO,gravidade,338367,96.44,2654,9829,350850


Distribuição das duas colunas de bloco avaliado, com a prevalência de óbito em cada valor.

In [5]:
linhas = []
for col, bloco, n_campos in [(BLOCO_ALARME, 'alarme', len(ALRM_COLS)),
                             (BLOCO_GRAVIDADE, 'gravidade', len(GRAV_COLS))]:
    for valor in [0, 1]:
        m = base_corrigida[col] == valor
        sub = base_corrigida[m]
        linhas.append({
            'coluna': col,
            'bloco': bloco,
            'n_campos_do_bloco': n_campos,
            'valor': valor,
            'n': int(m.sum()),
            'pct': round(m.mean() * 100, 3),
            'n_obitos': int((sub['target'] == 1).sum()),
            'prevalencia_obito_pct': round((sub['target'] == 1).mean() * 100, 3) if m.sum() else np.nan,
            'n_base': N,
            'prevalencia_coorte_pct': round(n_obito / N * 100, 3),
        })

blocos_dist = pd.DataFrame(linhas)
salvar_tab(blocos_dist, 'distribuicao_colunas_bloco_avaliado.csv')
display(blocos_dist)

[tabela] analysis/results_codificacao/distribuicao_colunas_bloco_avaliado.csv  (4 linhas)


,coluna,bloco,n_campos_do_bloco,valor,n,pct,n_obitos,prevalencia_obito_pct,n_base,prevalencia_coorte_pct
0,bloco_alarme_avaliado,alarme,9,0,242288,69.057,2288,0.944,350850,2.694
1,bloco_alarme_avaliado,alarme,9,1,108562,30.943,7163,6.598,350850,2.694
2,bloco_gravidade_avaliado,gravidade,15,0,338239,96.406,3271,0.967,350850,2.694
3,bloco_gravidade_avaliado,gravidade,15,1,12611,3.594,6180,49.005,350850,2.694


Reconstrói os indicadores de bloco como em `notebooks/06` e mede a concordância com as duas colunas novas.

In [6]:
ind_06 = {}
for c in ALRM_COLS + GRAV_COLS:
    codigo = pd.to_numeric(bruto[c].astype(str).str.strip(), errors='coerce')
    ind_06[f'aus_{c}'] = (~codigo.isin([1, 2])).astype('int8')
ind_06 = pd.DataFrame(ind_06, index=bruto.index)

bloco_06_alarme    = (ind_06[[f'aus_{c}' for c in ALRM_COLS]].sum(axis=1) < len(ALRM_COLS)).astype('int8')
bloco_06_gravidade = (ind_06[[f'aus_{c}' for c in GRAV_COLS]].sum(axis=1) < len(GRAV_COLS)).astype('int8')

linhas = []
for col, serie_06, bloco in [(BLOCO_ALARME, bloco_06_alarme, 'alarme'),
                             (BLOCO_GRAVIDADE, bloco_06_gravidade, 'gravidade')]:
    nova = base_corrigida[col].to_numpy()
    antiga_06 = serie_06.to_numpy()
    discord = int((nova != antiga_06).sum())
    linhas.append({
        'coluna': col,
        'bloco': bloco,
        'n_base': N,
        'n_positivos_treat_corrigida': int(nova.sum()),
        'n_positivos_reconstrucao_06': int(antiga_06.sum()),
        'n_discordantes': discord,
        'pct_discordantes': round(discord / N * 100, 4),
        'concordancia_total': bool(discord == 0),
    })

concord = pd.DataFrame(linhas)
salvar_tab(concord, 'concordancia_com_notebook_06.csv')
display(concord)

if bool(concord['concordancia_total'].all()):
    print('Concordância total: as duas colunas novas reproduzem exatamente os '
          'indicadores de bloco de notebooks/06.')
else:
    raise AssertionError('PARADA: divergência entre as colunas novas e os indicadores de 06.')

[tabela] analysis/results_codificacao/concordancia_com_notebook_06.csv  (2 linhas)


,coluna,bloco,n_base,n_positivos_treat_corrigida,n_positivos_reconstrucao_06,n_discordantes,pct_discordantes,concordancia_total
0,bloco_alarme_avaliado,alarme,350850,108562,108562,0,0.0,True
1,bloco_gravidade_avaliado,gravidade,350850,12611,12611,0,0.0,True


Concordância total: as duas colunas novas reproduzem exatamente os indicadores de bloco de notebooks/06.


Composição das três configurações, por grupo de campo.

In [7]:
CONFIGS = configuracoes.todas_configuracoes()

linhas = []
for nome, cfg in CONFIGS.items():
    preds = cfg['preditores']
    linhas.append({
        'configuracao': ROTULO_CONFIG[nome],
        'n_preditores': len(preds),
        'codificacao_treat': cfg['codificacao_treat'],
        'n_demograficos': sum(c in configuracoes.DEMOGRAFICOS for c in preds),
        'n_sintomas': sum(c in configuracoes.SINTOMAS for c in preds),
        'n_alrm': sum(c in ALRM_COLS for c in preds),
        'n_grav': sum(c in GRAV_COLS for c in preds),
        'n_derivados': sum(c in configuracoes.DERIVADOS for c in preds),
        'n_indicadores_bloco': sum(c in treat_data.BLOCO_COLS for c in preds),
        'tratamento_ausente_blocos': cfg['tratamento_ausente']['blocos_alrm_grav'],
        'descricao': cfg['descricao'],
        'n_base_coorte': N,
    })

resumo_cfg = pd.DataFrame(linhas)
salvar_tab(resumo_cfg, 'configuracoes_preditores.csv')
display(resumo_cfg[['configuracao', 'n_preditores', 'codificacao_treat',
                    'n_demograficos', 'n_sintomas', 'n_alrm', 'n_grav',
                    'n_derivados', 'n_indicadores_bloco', 'n_base_coorte']])

[tabela] analysis/results_codificacao/configuracoes_preditores.csv  (3 linhas)


,configuracao,n_preditores,codificacao_treat,n_demograficos,n_sintomas,n_alrm,n_grav,n_derivados,n_indicadores_bloco,n_base_coorte
0,original,51,antiga,5,20,9,15,2,0,350850
1,final,53,corrigida,4,20,9,15,2,2,350850
2,sem critérios de gravidade,37,corrigida,4,20,9,0,2,1,350850


Pertinência campo a campo: o que entra e o que fica de fora em cada configuração.

In [8]:
todos_campos = sorted(
    {c for cfg in CONFIGS.values() for c in cfg['preditores']},
    key=lambda c: (configuracoes.PREDITORES_ANTIGA + treat_data.BLOCO_COLS).index(c)
    if c in configuracoes.PREDITORES_ANTIGA + treat_data.BLOCO_COLS else 999
)

def grupo_do_campo(c):
    if c in configuracoes.DEMOGRAFICOS: return 'demografico'
    if c in configuracoes.SINTOMAS: return 'sintoma/comorbidade'
    if c in ALRM_COLS: return 'sinal de alarme'
    if c in GRAV_COLS: return 'criterio de gravidade'
    if c in treat_data.BLOCO_COLS: return 'indicador de bloco'
    if c in configuracoes.DERIVADOS: return 'derivado'
    return 'outro'

pert = pd.DataFrame({'campo': todos_campos})
pert['grupo'] = pert['campo'].map(grupo_do_campo)
for nome, cfg in CONFIGS.items():
    pert[ROTULO_CONFIG[nome]] = pert['campo'].isin(cfg['preditores']).astype(int)
pert['n_base_campos'] = len(todos_campos)

salvar_tab(pert, 'pertinencia_campos_por_configuracao.csv')
display(pert)

preds_notif = CONFIGS['notificacao']['preditores']
fora_notif = [c for c in todos_campos if c not in preds_notif]
print(f'\n{ROTULO_CONFIG["notificacao"]}: {len(preds_notif)} preditores de '
      f'{len(todos_campos)} campos')
print(f'  entraram: {", ".join(preds_notif)}')
print(f'  ficaram de fora ({len(fora_notif)}): {", ".join(fora_notif)}')

[tabela] analysis/results_codificacao/pertinencia_campos_por_configuracao.csv  (54 linhas)


,campo,grupo,original,final,sem critérios de gravidade,n_base_campos
0,CS_SEXO,demografico,1,1,1,54
1,CS_GESTANT,demografico,1,1,1,54
2,CS_RACA,demografico,1,1,1,54
3,CS_ESCOL_N,demografico,1,1,1,54
4,SG_UF,demografico,1,0,0,54
5,FEBRE,sintoma/comorbidade,1,1,1,54
6,MIALGIA,sintoma/comorbidade,1,1,1,54
7,CEFALEIA,sintoma/comorbidade,1,1,1,54
8,EXANTEMA,sintoma/comorbidade,1,1,1,54
9,VOMITO,sintoma/comorbidade,1,1,1,54



sem critérios de gravidade: 37 preditores de 54 campos
  entraram: CS_SEXO, CS_GESTANT, CS_RACA, CS_ESCOL_N, macrorregiao, FEBRE, MIALGIA, CEFALEIA, EXANTEMA, VOMITO, NAUSEA, DOR_COSTAS, CONJUNTVIT, ARTRITE, ARTRALGIA, PETEQUIA_N, LEUCOPENIA, LACO, DOR_RETRO, DIABETES, HEMATOLOG, HEPATOPAT, RENAL, HIPERTENSA, AUTO_IMUNE, ALRM_HIPOT, ALRM_PLAQ, ALRM_VOM, ALRM_SANG, ALRM_HEMAT, ALRM_ABDOM, ALRM_LETAR, ALRM_HEPAT, ALRM_LIQ, age_years, epi_week, bloco_alarme_avaliado
  ficaram de fora (17): SG_UF, GRAV_PULSO, GRAV_CONV, GRAV_ENCH, GRAV_INSUF, GRAV_TAQUI, GRAV_EXTRE, GRAV_HIPOT, GRAV_HEMAT, GRAV_MELEN, GRAV_METRO, GRAV_SANG, GRAV_AST, GRAV_MIOC, GRAV_CONSC, GRAV_ORGAO, bloco_gravidade_avaliado


Códigos da escolaridade por partição: os 0 a 8 formam a escala ordinal de níveis crescentes, e o 10, "não se aplica", não pertence a ela.

In [9]:
PARTICOES_ANOS = {
    'treino': split_data.TRAIN_YEARS,
    'teste': split_data.TEST_YEARS,
    'validacao': [2025],
}
CODIGO_NAO_SE_APLICA = preprocessamento.CODIGO_ESCOLARIDADE_NAO_SE_APLICA

linhas = []
for particao, anos in PARTICOES_ANOS.items():
    m = base_corrigida['year'].isin(anos) & base_corrigida['target'].notna()
    sub = base_corrigida.loc[m]
    esc = sub['CS_ESCOL_N']
    n_informada = int(esc.notna().sum())
    m_10 = esc == CODIGO_NAO_SE_APLICA
    n_10 = int(m_10.sum())
    linhas.append({
        'particao': particao,
        'anos': '-'.join(str(a) for a in anos),
        'n_base': len(sub),
        'n_escolaridade_informada': n_informada,
        'n_codigo_10': n_10,
        'pct_da_base': round(n_10 / len(sub) * 100, 1),
        'pct_dos_informados': round(n_10 / n_informada * 100, 1),
        'obitos_codigo_10': int((sub.loc[m_10, 'target'] == 1).sum()),
        'prevalencia_obito_codigo_10_pct': round(
            float((sub.loc[m_10, 'target'] == 1).mean()) * 100, 2),
        'prevalencia_obito_particao_pct': round(
            float((sub['target'] == 1).mean()) * 100, 2),
    })

escolaridade_10 = pd.DataFrame(linhas)
salvar_tab(escolaridade_10, 'escolaridade_codigo_10_por_particao.csv')
display(escolaridade_10)

dist_esc = (base_corrigida['CS_ESCOL_N'].value_counts(dropna=False)
            .rename_axis('codigo').reset_index(name='n').sort_values('codigo'))
dist_esc['pct'] = (dist_esc['n'] / N * 100).round(1)
print(f'\nNa coorte inteira: {n_(int((base_corrigida["CS_ESCOL_N"] == CODIGO_NAO_SE_APLICA).sum()))} '
      f'registros no código 10, de {n_(int(base_corrigida["CS_ESCOL_N"].notna().sum()))} com '
      f'escolaridade informada.')
print('Códigos presentes:', sorted(c for c in dist_esc['codigo'] if pd.notna(c)))

[tabela] analysis/results_codificacao/escolaridade_codigo_10_por_particao.csv  (3 linhas)


,particao,anos,n_base,n_escolaridade_informada,n_codigo_10,pct_da_base,pct_dos_informados,obitos_codigo_10,prevalencia_obito_codigo_10_pct,prevalencia_obito_particao_pct
0,treino,2020-2021-2022-2023,127074,73360,13080,10.3,17.8,96,0.73,2.06
1,teste,2024,160302,79464,12861,8.0,16.2,95,0.74,3.30
2,validacao,2025,63066,36252,5210,8.3,14.4,24,0.46,2.43



Na coorte inteira: 31.238 registros no código 10, de 189.358 com escolaridade informada.
Códigos presentes: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0]


Idade dos registros do código 10 e dos que declaram escolaridade, por partição: posição, cauda, proporções abaixo de 7 e de 8 anos completos, e a associação no sentido inverso.

In [10]:
COL_IDADE = 'age_years'


def descrever_idade(idade, n_registros):
    '''Posição, cauda e proporções abaixo de 7 e de 8 anos completos.

    Os dois cortes andam juntos porque separam duas leituras: 7 anos é onde o corte estaria
    se a categoria fosse etária no sentido estrito, e 8 é onde a coorte de fato o coloca.
    '''
    a = idade.dropna()
    q1, mediana, q3 = a.quantile([0.25, 0.50, 0.75])
    p90, p95, p99 = a.quantile([0.90, 0.95, 0.99])
    return {
        'n': n_registros,
        'n_com_idade': len(a),
        'mediana': round(float(mediana), 1),
        'q1': round(float(q1), 1),
        'q3': round(float(q3), 1),
        'iqr': round(float(q3 - q1), 1),
        'p90': round(float(p90), 1),
        'p95': round(float(p95), 1),
        'p99': round(float(p99), 1),
        'idade_maxima': round(float(a.max()), 1),
        'n_abaixo_de_7': int((a < 7).sum()),
        'pct_abaixo_de_7': round(float((a < 7).mean()) * 100, 1),
        'pct_abaixo_de_8': round(float((a < 8).mean()) * 100, 1),
    }


linhas, inversa = [], []
for particao, anos in PARTICOES_ANOS.items():
    m = base_corrigida['year'].isin(anos) & base_corrigida['target'].notna()
    sub = base_corrigida.loc[m]
    esc, idade = sub['CS_ESCOL_N'], sub[COL_IDADE]
    m_10 = esc == CODIGO_NAO_SE_APLICA
    m_informada = esc.notna() & ~m_10

    for grupo, m_grupo in (('código 10', m_10), ('informada (0 a 8)', m_informada)):
        linhas.append({'particao': particao, 'grupo': grupo,
                       **descrever_idade(idade[m_grupo], int(m_grupo.sum()))})

    # Sentido inverso: dos que têm menos de 7 anos, quantos trazem o código 10, e o que
    # declaram os que não trazem.
    m_abaixo = idade < 7
    demais = esc[m_abaixo & ~m_10]
    declarados = demais.dropna().astype(int).value_counts().sort_index()
    m_residuo = m_10 & (idade >= 8)
    inversa.append({
        'particao': particao,
        'n_abaixo_de_7': int(m_abaixo.sum()),
        'n_abaixo_de_7_com_codigo_10': int((m_abaixo & m_10).sum()),
        'pct_abaixo_de_7_com_codigo_10': round(
            float((m_abaixo & m_10).sum()) / int(m_abaixo.sum()) * 100, 1),
        'n_abaixo_de_7_com_outra_escolaridade': int(demais.notna().sum()),
        'codigos_declarados_pelos_demais': (
            '; '.join(f'{c}: {n}' for c, n in declarados.items()) or '—'),
        'n_abaixo_de_7_sem_escolaridade': int(demais.isna().sum()),
        'n_codigo_10_com_7_anos': int((m_10 & (idade >= 7) & (idade < 8)).sum()),
        'n_codigo_10_com_8_ou_mais': int(m_residuo.sum()),
        'pct_codigo_10_com_8_ou_mais': round(
            float(m_residuo.sum()) / int(m_10.sum()) * 100, 1),
        'idade_maxima_codigo_10': round(float(idade[m_10].max()), 1),
    })

idade_escolaridade = pd.DataFrame(linhas)
codigo_10_idade = pd.DataFrame(inversa)
salvar_tab(idade_escolaridade, 'idade_por_grupo_escolaridade.csv')
salvar_tab(codigo_10_idade, 'codigo_10_e_idade.csv')
display(idade_escolaridade)
display(codigo_10_idade)

ie = idade_escolaridade.set_index(['particao', 'grupo'])
for particao in PARTICOES_ANOS:
    c10, inf = ie.loc[(particao, 'código 10')], ie.loc[(particao, 'informada (0 a 8)')]
    r = codigo_10_idade.set_index('particao').loc[particao]
    print(f'\n{particao}')
    print(f'  código 10 ({n_(c10["n"])}): mediana {v_(c10["mediana"])} anos, IQI de '
          f'{v_(c10["q1"])} a {v_(c10["q3"])}; p90 {v_(c10["p90"])}, p95 {v_(c10["p95"])}, '
          f'p99 {v_(c10["p99"])}; máxima {v_(c10["idade_maxima"], 0)}; '
          f'{v_(c10["pct_abaixo_de_7"])}% abaixo de 7 anos e {v_(c10["pct_abaixo_de_8"])}% '
          f'abaixo de 8')
    print(f'  informada ({n_(inf["n"])}): mediana {v_(inf["mediana"])} anos, IQI de '
          f'{v_(inf["q1"])} a {v_(inf["q3"])}; p90 {v_(inf["p90"])}, p95 {v_(inf["p95"])}, '
          f'p99 {v_(inf["p99"])}; máxima {v_(inf["idade_maxima"], 0)}; '
          f'{v_(inf["pct_abaixo_de_7"])}% abaixo de 7 anos ({n_(inf["n_abaixo_de_7"])} '
          f'registros)')
    print(f'  abaixo de 7 anos: {v_(r["pct_abaixo_de_7_com_codigo_10"])}% trazem o código 10 '
          f'({n_(r["n_abaixo_de_7_com_codigo_10"])} de {n_(r["n_abaixo_de_7"])}); dos '
          f'{n_(r["n_abaixo_de_7"] - r["n_abaixo_de_7_com_codigo_10"])} restantes, '
          f'{n_(r["n_abaixo_de_7_com_outra_escolaridade"])} declaram outro código '
          f'({r["codigos_declarados_pelos_demais"]}) e '
          f'{n_(r["n_abaixo_de_7_sem_escolaridade"])} não têm escolaridade registrada')
    print(f'  com exatamente 7 anos: {n_(r["n_codigo_10_com_7_anos"])} trazem o código 10; '
          f'com 8 anos ou mais: {n_(r["n_codigo_10_com_8_ou_mais"])} '
          f'({v_(r["pct_codigo_10_com_8_ou_mais"])}%), até {v_(r["idade_maxima_codigo_10"], 0)} '
          f'anos')


[tabela] analysis/results_codificacao/idade_por_grupo_escolaridade.csv  (6 linhas)
[tabela] analysis/results_codificacao/codigo_10_e_idade.csv  (3 linhas)


,particao,grupo,n,n_com_idade,mediana,q1,q3,iqr,p90,p95,p99,idade_maxima,n_abaixo_de_7,pct_abaixo_de_7,pct_abaixo_de_8
0,treino,código 10,13080,13078,4.0,1.0,6.0,5.0,7.0,7.0,7.0,88.0,11029,84.3,99.5
1,treino,informada (0 a 8),60280,60275,39.0,23.0,59.0,36.0,74.0,80.0,89.0,108.0,16,0.0,0.1
2,teste,código 10,12861,12860,4.0,1.0,6.0,5.0,7.0,7.0,7.0,88.0,10947,85.1,99.0
3,teste,informada (0 a 8),66603,66598,44.0,26.0,64.0,38.0,77.0,83.0,91.0,110.0,8,0.0,0.0
4,validacao,código 10,5210,5210,4.0,1.0,6.0,5.0,7.0,7.0,7.0,87.0,4493,86.2,99.8
5,validacao,informada (0 a 8),31042,31042,43.0,26.0,61.0,35.0,75.0,81.0,90.0,114.0,7,0.0,0.1


,particao,n_abaixo_de_7,n_abaixo_de_7_com_codigo_10,pct_abaixo_de_7_com_codigo_10,n_abaixo_de_7_com_outra_escolaridade,codigos_declarados_pelos_demais,n_abaixo_de_7_sem_escolaridade,n_codigo_10_com_7_anos,n_codigo_10_com_8_ou_mais,pct_codigo_10_com_8_ou_mais,idade_maxima_codigo_10
0,treino,11075,11029,99.6,16,0: 3; 1: 13,30,1978,71,0.5,88.0
1,teste,10979,10947,99.7,8,0: 1; 1: 6; 3: 1,24,1787,126,1.0,88.0
2,validacao,4517,4493,99.5,7,1: 6; 2: 1,17,705,12,0.2,87.0



treino
  código 10 (13.080): mediana 4,0 anos, IQI de 1,0 a 6,0; p90 7,0, p95 7,0, p99 7,0; máxima 88; 84,3% abaixo de 7 anos e 99,5% abaixo de 8
  informada (60.280): mediana 39,0 anos, IQI de 23,0 a 59,0; p90 74,0, p95 80,0, p99 89,0; máxima 108; 0,0% abaixo de 7 anos (16 registros)
  abaixo de 7 anos: 99,6% trazem o código 10 (11.029 de 11.075); dos 46 restantes, 16 declaram outro código (0: 3; 1: 13) e 30 não têm escolaridade registrada
  com exatamente 7 anos: 1.978 trazem o código 10; com 8 anos ou mais: 71 (0,5%), até 88 anos

teste
  código 10 (12.861): mediana 4,0 anos, IQI de 1,0 a 6,0; p90 7,0, p95 7,0, p99 7,0; máxima 88; 85,1% abaixo de 7 anos e 99,0% abaixo de 8
  informada (66.603): mediana 44,0 anos, IQI de 26,0 a 64,0; p90 77,0, p95 83,0, p99 91,0; máxima 110; 0,0% abaixo de 7 anos (8 registros)
  abaixo de 7 anos: 99,7% trazem o código 10 (10.947 de 10.979); dos 32 restantes, 8 declaram outro código (0: 1; 1: 6; 3: 1) e 24 não têm escolaridade registrada
  com exatam

Grava as bases das configurações final e sem critérios de gravidade em `data/processed/`. `dengue_treated_antiga.parquet` não é regenerado: é o artefato congelado que reproduz os modelos publicados, e a célula confere que ele continua compatível com a lista de preditores da configuração em vez de reescrevê-lo.

In [11]:
BASE_POR_CODIFICACAO = {'antiga': base_antiga, 'corrigida': base_corrigida}
# SG_UF entra em COLUNAS_CARREGADAS: nas configurações em que deixou de ser preditora
# ela ainda acompanha a base, e nas em que continua preditora não pode entrar duas vezes.
EXTRAS = configuracoes.COLUNAS_NAO_PREDITORAS + configuracoes.COLUNAS_CARREGADAS

# A configuração original é a linha de base da análise de referência, e o arquivo dela é a
# âncora da reprodução verificada em notebooks/09. Regenerá-lo com a correção do passo 6 trocaria
# uma reprodução conferida por outra a conferir, sem ganho: o OrdinalEncoder do sexo trata
# o texto 'nan' como desconhecido e o converte em NaN, então os dois caminhos convergem
# antes do estimador. Fica congelado, e a célula abaixo o lê em vez de escrever.
CONGELADAS = {'antiga'}

gravadas = []
bases = {}
for nome, cfg in CONFIGS.items():
    origem = BASE_POR_CODIFICACAO[cfg['codificacao_treat']]
    preds = [c for c in cfg['preditores'] if c in origem.columns]
    faltando = [c for c in cfg['preditores'] if c not in origem.columns]
    colunas = preds + [c for c in EXTRAS if c in origem.columns and c not in preds]

    destino = os.path.join(PROC_DIR, f'dengue_treated_{nome}.parquet')
    if nome in CONGELADAS:
        assert os.path.exists(destino), f'{nome}: arquivo congelado ausente — {destino}'
        base = pd.read_parquet(destino, engine='pyarrow')
        assert list(base.columns) == colunas, (
            f'{nome}: o arquivo congelado tem outras colunas que as da configuração')
        situacao = 'congelado, não regenerado'
    else:
        base = origem[colunas].copy()
        base.to_parquet(destino, index=False, engine='pyarrow')
        situacao = 'regravado'
    bases[nome] = base

    gravadas.append({
        'configuracao': ROTULO_CONFIG[nome],
        'arquivo': os.path.relpath(destino, ROOT),
        'situacao': situacao,
        'sha256': hashlib.sha256(open(destino, 'rb').read()).hexdigest()[:16],
        'n_linhas': len(base),
        'n_colunas': base.shape[1],
        'n_preditores': len(preds),
        'n_sexo_ausente_isna': int(base['CS_SEXO'].isna().sum()),
        'n_sexo_ausente_texto': int((base['CS_SEXO'].astype(str) == 'nan').sum()),
        'colunas_nao_preditoras': ', '.join(c for c in EXTRAS
                                            if c in origem.columns and c not in preds),
        'preditores_ausentes_na_origem': ', '.join(faltando) if faltando else '—',
    })
    print(f'{ROTULO_CONFIG[nome]:{LARG_ROTULO}s} -> {os.path.relpath(destino, ROOT)}  '
          f'({len(base):,} x {base.shape[1]}; {len(preds)} preditores) [{situacao}]')
    if faltando:
        print(f'  [AVISO] preditores da configuração ausentes na base tratada: {faltando}')

gravadas = pd.DataFrame(gravadas)
gravadas['n_base_coorte'] = N
salvar_tab(gravadas, 'bases_geradas.csv')
display(gravadas[['configuracao', 'arquivo', 'situacao', 'n_linhas', 'n_colunas',
                  'n_sexo_ausente_isna', 'n_sexo_ausente_texto']])

restantes = sorted(f for f in os.listdir(PROC_DIR)
                   if f.startswith('dengue_treated_') and f.endswith('.parquet')
                   and f[len('dengue_treated_'):-len('.parquet')] not in CONFIGS)
print(f'\nParquet por configuração em {os.path.relpath(PROC_DIR, ROOT)}: '
      f'{len(CONFIGS)} esperados, {len(restantes)} de configurações inexistentes '
      f'({", ".join(restantes) if restantes else "nenhum"}).')
print(f'A base atual ({os.path.relpath(TREATED_ATUAL, ROOT)}) não foi tocada.')

original                   -> data/processed/dengue_treated_antiga.parquet  (350,850 x 53; 51 preditores) [congelado, não regenerado]


final                      -> data/processed/dengue_treated_corrigida.parquet  (350,850 x 56; 53 preditores) [regravado]


sem critérios de gravidade -> data/processed/dengue_treated_notificacao.parquet  (350,850 x 40; 37 preditores) [regravado]
[tabela] analysis/results_codificacao/bases_geradas.csv  (3 linhas)


,configuracao,arquivo,situacao,n_linhas,n_colunas,n_sexo_ausente_isna,n_sexo_ausente_texto
0,original,data/processed/dengue_treated_antiga.parquet,"congelado, não regenerado",350850,53,2,187
1,final,data/processed/dengue_treated_corrigida.parquet,regravado,350850,56,189,189
2,sem critérios de gravidade,data/processed/dengue_treated_notificacao.parquet,regravado,350850,40,189,189



Parquet por configuração em data/processed: 3 esperados, 0 de configurações inexistentes (nenhum).
A base atual (data/processed/dengue_treated.parquet) não foi tocada.


Confere que a correção do passo 6 não move a matriz de desenho: compara, por sha256, a matriz de cada família com o ausente do sexo em NaN e com ele no texto `'nan'` de antes.

In [12]:
def sha256_matriz(Z):
    '''Impressão digital da matriz de desenho, em ordem de memória e dtype fixo.'''
    return hashlib.sha256(
        np.ascontiguousarray(np.asarray(Z, dtype=np.float64)).tobytes()).hexdigest()


# Reconstrói o estado anterior à correção: o ausente do sexo como o texto que o astype(str)
# do passo 6 produzia. Tudo o mais é idêntico.
TEXTO_DE_AUSENTE = 'nan'

m_treino_cfg = (base_corrigida['year'].isin(split_data.TRAIN_YEARS)
                & base_corrigida['target'].notna())

linhas = []
for nome in [c for c in CONFIGS if c not in CONGELADAS]:
    base = bases[nome]
    preds = [c for c in configuracoes.preditores(nome) if c in base.columns]
    X = base.loc[m_treino_cfg.to_numpy(), preds]
    X_texto = X.copy()
    X_texto['CS_SEXO'] = X_texto['CS_SEXO'].where(X_texto['CS_SEXO'].notna(),
                                                  TEXTO_DE_AUSENTE)
    for familia in preprocessamento.FAMILIAS:
        ct, _ = preprocessamento.preprocessador(nome, familia)
        Z = ct.fit_transform(X)
        ct_texto, _ = preprocessamento.preprocessador(nome, familia)
        Z_texto = ct_texto.fit_transform(X_texto)
        nomes = list(ct.get_feature_names_out())
        linhas.append({
            'configuracao': ROTULO_CONFIG[nome],
            'familia': familia,
            'n_linhas': Z.shape[0],
            'n_colunas_desenho': Z.shape[1],
            'tem_indicadora_sexo_ausente': preprocessamento.COL_SEXO_AUSENTE in nomes,
            'n_sexo_ausente_na_entrada': int(X['CS_SEXO'].isna().sum()),
            'sha256_com_nan': sha256_matriz(Z)[:16],
            'sha256_com_texto': sha256_matriz(Z_texto)[:16],
            'matrizes_identicas': sha256_matriz(Z) == sha256_matriz(Z_texto),
        })
        del Z, Z_texto

matrizes = pd.DataFrame(linhas)
salvar_tab(matrizes, 'matriz_desenho_antes_e_depois_da_correcao.csv')
display(matrizes[['configuracao', 'familia', 'n_colunas_desenho',
                  'tem_indicadora_sexo_ausente', 'n_sexo_ausente_na_entrada',
                  'sha256_com_nan', 'sha256_com_texto', 'matrizes_identicas']])

if not bool(matrizes['matrizes_identicas'].all()):
    raise AssertionError('PARADA: a correção do passo 6 moveu alguma matriz de desenho.')
print('As matrizes de desenho são idênticas por sha256 nas duas formas do ausente: a '
      'indicadora de sexo testa pertencimento a {F, M} e o codificador ordinal trata os '
      'dois casos como desconhecido. O que muda é a contagem a montante, por isna().')

[tabela] analysis/results_codificacao/matriz_desenho_antes_e_depois_da_correcao.csv  (6 linhas)


,configuracao,familia,n_colunas_desenho,tem_indicadora_sexo_ausente,n_sexo_ausente_na_entrada,sha256_com_nan,sha256_com_texto,matrizes_identicas
0,final,linear,71,True,66,ee9b63a4bbbb1d19,ee9b63a4bbbb1d19,True
1,final,arvore,70,False,66,7e261052a9f362a6,7e261052a9f362a6,True
2,final,boosting,70,False,66,dfdde23654620d32,dfdde23654620d32,True
3,sem critérios de gravidade,linear,55,True,66,2f09f3686033826b,2f09f3686033826b,True
4,sem critérios de gravidade,arvore,54,False,66,b1f660fba798fad3,b1f660fba798fad3,True
5,sem critérios de gravidade,boosting,54,False,66,5ffe6d190d2145ce,5ffe6d190d2145ce,True


As matrizes de desenho são idênticas por sha256 nas duas formas do ausente: a indicadora de sexo testa pertencimento a {F, M} e o codificador ordinal trata os dois casos como desconhecido. O que muda é a contagem a montante, por isna().


Três medidas de ausência por configuração e o EPP na janela de treino.

In [13]:
m_treino = base_corrigida['year'].isin(split_data.TRAIN_YEARS)
m_teste  = base_corrigida['year'].isin(split_data.TEST_YEARS)
n_treino = int(m_treino.sum())
eventos_treino = int((base_corrigida.loc[m_treino, 'target'] == 1).sum())

print(f'Treino {split_data.TRAIN_YEARS}: {n_treino:,} registros | {eventos_treino:,} óbitos '
      f'({eventos_treino / n_treino * 100:.2f}%)')
print(f'Teste  {split_data.TEST_YEARS}: {int(m_teste.sum()):,} registros | '
      f'{int((base_corrigida.loc[m_teste, "target"] == 1).sum()):,} óbitos')
print(f"config.json publicado: train_size={cfg_publicado['train_size']:,} "
      f"train_events={cfg_publicado['train_events']:,}")

# Tres medidas de ausencia, porque elas nao sao a mesma coisa:
#   - na base gravada em disco;
#   - restrita aos campos clinicos ALRM_*/GRAV_* presentes na configuracao;
#   - no que chega ao estimador, depois do esquema declarado para o bloco
#     alrm_grav (constante 0 na antiga, NaN preservado nas demais).
linhas = []
for nome, cfg in CONFIGS.items():
    base = bases[nome]
    preds = [c for c in cfg['preditores'] if c in base.columns]
    X = base[preds]
    n_cel = X.shape[0] * X.shape[1]
    n_aus = int(X.isna().sum().sum())

    campos_clinicos = [c for c in preds if c in set(ALRM_COLS) | set(GRAV_COLS)]
    n_cel_cl = X.shape[0] * len(campos_clinicos)
    n_aus_cl = int(X[campos_clinicos].isna().sum().sum()) if campos_clinicos else 0

    # aplica o esquema declarado apenas ao bloco alrm_grav
    X_modelo = X.copy()
    if cfg['codificacao_treat'] == 'antiga' and campos_clinicos:
        X_modelo[campos_clinicos] = X_modelo[campos_clinicos].fillna(0)
    n_aus_modelo = int(X_modelo.isna().sum().sum())
    n_aus_cl_modelo = (int(X_modelo[campos_clinicos].isna().sum().sum())
                       if campos_clinicos else 0)

    Xtr = X[m_treino.to_numpy()]
    n_cel_tr = Xtr.shape[0] * Xtr.shape[1]
    n_aus_tr = int(Xtr.isna().sum().sum())

    linhas.append({
        'configuracao': ROTULO_CONFIG[nome],
        'n_preditores': len(preds),
        'n_linhas': len(base),
        'n_celulas': n_cel,
        'n_ausentes_base': n_aus,
        'pct_ausentes_base': round(n_aus / n_cel * 100, 2),
        'pct_ausentes_base_treino': round(n_aus_tr / n_cel_tr * 100, 2),
        'n_campos_clinicos': len(campos_clinicos),
        'pct_ausentes_campos_clinicos': round(n_aus_cl / n_cel_cl * 100, 2) if campos_clinicos else np.nan,
        'pct_ausentes_campos_clinicos_no_modelo': round(n_aus_cl_modelo / n_cel_cl * 100, 2) if campos_clinicos else np.nan,
        'pct_ausentes_no_modelo': round(n_aus_modelo / n_cel * 100, 2),
        'n_preditores_com_algum_ausente': int((X.isna().sum() > 0).sum()),
        'n_treino': n_treino,
        'eventos_treino': eventos_treino,
        'epp_treino': round(eventos_treino / len(preds), 2),
    })

comparativo = pd.DataFrame(linhas)
salvar_tab(comparativo, 'comparativo_configuracoes.csv')
display(comparativo)

Treino [2020, 2021, 2022, 2023]: 127,074 registros | 2,624 óbitos (2.06%)
Teste  [2024]: 160,302 registros | 5,292 óbitos
config.json publicado: train_size=127,074 train_events=2,624


[tabela] analysis/results_codificacao/comparativo_configuracoes.csv  (3 linhas)


,configuracao,n_preditores,n_linhas,n_celulas,n_ausentes_base,pct_ausentes_base,pct_ausentes_base_treino,n_campos_clinicos,pct_ausentes_campos_clinicos,pct_ausentes_campos_clinicos_no_modelo,pct_ausentes_no_modelo,n_preditores_com_algum_ausente,n_treino,eventos_treino,epp_treino
0,original,51,350850,17893350,7545127,42.17,44.11,24,86.25,0.00,1.58,49,127074,2624,51.45
1,final,53,350850,18595050,7545314,40.58,42.44,24,86.25,86.25,40.58,49,127074,2624,49.51
2,sem critérios de gravidade,37,350850,12981450,2469247,19.02,21.37,9,69.25,69.25,19.02,34,127074,2624,70.92


Recorte de EPP e ausência para a tabela do resumo, com o EPP publicado como referência.

In [14]:
epp = comparativo[['configuracao', 'n_preditores', 'n_treino', 'eventos_treino',
                   'epp_treino', 'pct_ausentes_base', 'pct_ausentes_no_modelo',
                   'pct_ausentes_campos_clinicos', 'pct_ausentes_campos_clinicos_no_modelo']].copy()
epp['eventos_por_preditor_publicado'] = round(cfg_publicado['train_events'] / 51, 2)
salvar_tab(epp, 'epp_por_configuracao.csv')
display(epp)

for _, r in epp.iterrows():
    print(f'{r["configuracao"]:{LARG_ROTULO}s} {int(r["n_preditores"]):>3} preditores  '
          f'EPP = {r["eventos_treino"]:,} / {int(r["n_preditores"])} = {r["epp_treino"]:.2f}')

[tabela] analysis/results_codificacao/epp_por_configuracao.csv  (3 linhas)


,configuracao,n_preditores,n_treino,eventos_treino,epp_treino,pct_ausentes_base,pct_ausentes_no_modelo,pct_ausentes_campos_clinicos,pct_ausentes_campos_clinicos_no_modelo,eventos_por_preditor_publicado
0,original,51,127074,2624,51.45,42.17,1.58,86.25,0.00,51.45
1,final,53,127074,2624,49.51,40.58,40.58,86.25,86.25,51.45
2,sem critérios de gravidade,37,127074,2624,70.92,19.02,19.02,69.25,69.25,51.45


original                    51 preditores  EPP = 2,624 / 51 = 51.45
final                       53 preditores  EPP = 2,624 / 53 = 49.51
sem critérios de gravidade  37 preditores  EPP = 2,624 / 37 = 70.92


## Conclusões

- A fusão de ausente com "não" não está em `utils/treat_data.py`: o módulo devolve os 24 campos com ausente como `NaN`.

- O parâmetro `codificacao` preserva o comportamento atual no padrão: as 63 colunas da base antiga têm os mesmos dtypes e as mesmas 350.850 linhas de `data/processed/dengue_treated.parquet`, e 62 delas são idênticas célula a célula.

- A única divergência é a correção do passo 6, em `CS_SEXO`: 187 células que já estavam ausentes e que o `astype(str)` convertia no texto `'nan'`.
  Nenhum valor presente mudou, e o ausente detectável por `isna()` passa de 2 para 189.

- A correção não move a matriz de desenho: nas 6 combinações de configuração e família fora da original, a matriz com o ausente em `NaN` e a matriz com o ausente no texto têm o mesmo sha256.
  A indicadora `sexo_ausente` testa pertencimento a {F, M} e o codificador ordinal trata as duas formas como desconhecido.

- As duas colunas novas coincidem integralmente com os indicadores de bloco do notebook 06: 0 registros discordantes em 350.850.

- O bloco de alarme foi avaliado em 108.562 fichas, 30,9%, com prevalência de óbito de 6,6%, contra 0,9% nas 242.288 fichas não avaliadas.
  O bloco de gravidade foi avaliado em 12.611 fichas, 3,6%, com prevalência de 49,0%, contra 1,0% nas 338.239 não avaliadas.

- Nos 24 campos clínicos, 86,2% das células estão vazias na base tratada, nas duas codificações.
  O esquema declarado da configuração original reduz isso a 0,0% antes do estimador, preenchendo com 0; na final esses 86,2% chegam como ausentes e a disponibilidade do bloco passa a viajar nas duas colunas novas.

- Considerando todos os preditores, o que chega ao estimador tem 1,6% de células vazias na configuração original e 40,6% na final: a diferença é o que a imputação com 0 vinha ocultando.

- A configuração sem critérios de gravidade ficou com 37 preditores: os 51 da codificação publicada menos os 15 critérios de gravidade, mais o indicador de bloco de alarme avaliado.
  Os 9 sinais de alarme que ela conserva chegam ao estimador com 69,2% de células ausentes.

- O EPP no treino cai de 51,5 na configuração original, com 51 preditores, para 49,5 na final, com 53, e sobe para 70,9 na sem critérios de gravidade, com 37, sempre sobre 2.624 óbitos em 127.074 registros de treino.

- O código 10 da escolaridade, "não se aplica", atinge 13.080 registros no treino, 17,8% dos que têm escolaridade informada, 12.861 no teste de 2024, 16,2%, e 5.210 na validação de 2025, 14,4%.
  Nas configurações final e sem critérios de gravidade ele sai da escala ordinal e vira coluna indicadora própria; nas bases gravadas aqui o código segue bruto, porque a separação acontece em `utils/preprocessamento.py` e não nas bases.

- A idade de quem traz o código 10 tem mediana de 4,0 anos nas três partições, com intervalo interquartil de 1,0 a 6,0 no treino, contra medianas de 39,0, 44,0 e 43,0 entre os que declaram escolaridade.

- O corte observado está em 8 anos completos, e não em 7: os registros com exatamente 7 anos trazem o código, 1.978 no treino, 1.787 no teste e 705 na validação.
  Por isso a proporção abaixo de 7 anos fica em 84,3%, 85,1% e 86,2%, enquanto a proporção abaixo de 8 vai a 99,5%, 99,0% e 99,8%.

- No sentido inverso a associação é quase completa: 99,6% dos registros com menos de 7 anos no treino trazem o código 10, 99,7% no teste e 99,5% na validação.
  Os poucos que não trazem declaram códigos do início da escala e não níveis incompatíveis com a idade.

- Acima de 8 anos restam 71 registros com o código no treino, 0,5%, 126 no teste, 1,0%, e 12 na validação, 0,2%, com idade máxima de 88, 88 e 87 anos.
  É preenchimento inconsistente da ficha, não uma segunda população dentro da categoria.

- As bases das configurações final e sem critérios de gravidade foram regravadas em `data/processed/`, cada uma com 350.850 linhas e com o ausente de `CS_SEXO` em `NaN` de verdade.
  `dengue_treated_antiga.parquet` não foi regenerada, por ser o artefato congelado que reproduz os modelos publicados, e `data/processed/dengue_treated.parquet` não foi sobrescrita.